# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khaled-dragon/ML-intern/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
%pip -q install duckdb huggingface_hub
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"}

data_rich = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']} WHERE month='2026-03'),
    prior AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY
                        THEN gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY
                        THEN gsc_clicks ELSE 0 END) AS clk_prev30,
               AVG(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY
                        THEN gsc_avg_position END) AS pos_prev30,
               SUM(CASE WHEN report_date <= b.end_d - INTERVAL 30 DAY
                        AND report_date > b.end_d - INTERVAL 60 DAY
                        THEN gsc_impressions ELSE 0 END) AS imp_prev60_30,
               STDDEV(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY
                        THEN gsc_avg_position END) AS pos_volatility
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.month IN ('2026-02', '2026-03')
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100 AND imp_prev60_30 > 0
    ),
    outcome AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS imp_next30
        FROM {TABLES['fact_daily']} WHERE month = '2026-04'
        GROUP BY 1
    )
    SELECT p.*, o.imp_next30
    FROM prior p JOIN outcome o USING (content_hash_id)
""").df()

data_rich['is_declining'] = (data_rich['imp_next30'] < 0.8 * data_rich['imp_prev30']).astype(int)
data_rich['ctr_prev30'] = data_rich['clk_prev30'] / data_rich['imp_prev30']
data_rich['momentum_ratio'] = data_rich['imp_prev30'] / data_rich['imp_prev60_30']
print(f"{len(data_rich):,} rows, {data_rich['client_hash_id'].nunique()} clients")

Paste your HF READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

86,394 rows, 37 clients


In [2]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

feature_cols = ['imp_prev30', 'pos_prev30', 'ctr_prev30', 'momentum_ratio', 'pos_volatility']
X = data_rich[feature_cols].fillna(0)
y = data_rich['is_declining']

def precision_at_k(scores, y_true, k):
    order = scores.argsort()[::-1][:k]
    return y_true.iloc[order].mean()

# BEFORE: naive random row split (ignores client grouping)
X_tr_naive, X_te_naive, y_tr_naive, y_te_naive = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
rf_naive = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced').fit(X_tr_naive, y_tr_naive)
naive_scores = rf_naive.predict_proba(X_te_naive)[:, 1]

# AFTER: honest grouped split (same as Week 5)
gss = GroupShuffleSplit(test_size=0.25, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(data_rich, groups=data_rich['client_hash_id']))
train_r, test_r = data_rich.iloc[train_idx], data_rich.iloc[test_idx]
X_tr_g, y_tr_g = train_r[feature_cols].fillna(0), train_r['is_declining']
X_te_g, y_te_g = test_r[feature_cols].fillna(0), test_r['is_declining']
rf_grouped = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced').fit(X_tr_g, y_tr_g)
grouped_scores = rf_grouped.predict_proba(X_te_g)[:, 1]

comparison = pd.DataFrame({
    'split_type': ['BEFORE: naive random split', 'AFTER: grouped-by-client split'],
    'precision@20': [precision_at_k(naive_scores, y_te_naive, 20),
                      precision_at_k(grouped_scores, y_te_g, 20)],
    'precision@50': [precision_at_k(naive_scores, y_te_naive, 50),
                      precision_at_k(grouped_scores, y_te_g, 50)],
})
print(comparison)

                       split_type  precision@20  precision@50
0      BEFORE: naive random split           0.7          0.64
1  AFTER: grouped-by-client split           0.8          0.80


Interestingly, the grouped split scored higher (0.80) than the naive split
(0.64) at precision@50, the opposite of what I expected going in. This is
still a useful finding, not a contradiction: with only 37 clients total, a
handful of clients landing in the naive split's test set by chance can swing
the number either way. The real issue with the naive split isn't that it's
always optimistic, it's that it isn't measuring the same thing at all.
Letting one client's pages appear in both train and test means the model can
partly learn client-specific patterns, so its score doesn't tell you how it
would perform on a client it has never seen, which is the actual question a
capstone model needs to answer. The grouped split is the one whose number I'd
trust to generalize to a new client; the naive split's number, high or low,
isn't answering the right question.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
print("Feature audit — final model (w05 Attempt 3):\n")
for f in feature_cols:
    print(f"- {f}")

print("\nAudit checklist:")
print("1. imp_prev30, pos_prev30, ctr_prev30: aggregated only from days AFTER")
print("   (report_date > end_d - 30), i.e. strictly within the March feature window.")
print("2. momentum_ratio: uses imp_prev30 / imp_prev60_30, both windows end before April.")
print("3. pos_volatility: STDDEV computed only within the March 30-day window.")
print("4. Label (is_declining) is built from April's imp_next30 — a completely")
print("   separate month from all five features above. No overlap.")
print("5. fact_content_query_90d was tested and REJECTED (Week 5, Attempt 2):")
print("   its window (2026-04-02 to 2026-06-30) overlaps the outcome month directly.")

Feature audit — final model (w05 Attempt 3):

- imp_prev30
- pos_prev30
- ctr_prev30
- momentum_ratio
- pos_volatility

Audit checklist:
1. imp_prev30, pos_prev30, ctr_prev30: aggregated only from days AFTER
   (report_date > end_d - 30), i.e. strictly within the March feature window.
2. momentum_ratio: uses imp_prev30 / imp_prev60_30, both windows end before April.
3. pos_volatility: STDDEV computed only within the March 30-day window.
4. Label (is_declining) is built from April's imp_next30 — a completely
   separate month from all five features above. No overlap.
5. fact_content_query_90d was tested and REJECTED (Week 5, Attempt 2):
   its window (2026-04-02 to 2026-06-30) overlaps the outcome month directly.


All five features in the final model are computed strictly from report_date
values before April 1st (the outcome month), confirmed by the window
boundaries in the SQL itself. The one leakage risk I found (query-mix
features from fact_content_query_90d) was caught and dropped in Week 5,
because that table's fixed window overlaps the exact month used as the
label. No column derived from the label, or from any date on/after the
outcome window, appears among the final features.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original (a bolder version of a claim I actually made in Week 5)
"Stability of performance predicts future decline better than momentum,
proving that content teams should prioritize consistency over growth
signals."

### Rewritten (safe language)
"In this sample (86,394 rows, two months, one client panel), ctr_prev30 and
pos_volatility showed higher permutation importance than momentum_ratio for
predicting is_declining. This is an observed, directional pattern in this
specific dataset and time window, decision-support evidence for weighting
stability signals in a review queue, not a general or causal claim about
SEO performance."

**Why the rewrite matters:** the original uses "predicts" and "proving,"
words that imply a causal, generalizable law. The rewrite scopes the claim
to the exact data and window it came from, and swaps "predicts/proves" for
"observed/directional/decision-support" — language that matches what a
permutation-importance test on one sample can actually support.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.